In [0]:
%pip install openai mlflow==3.0.1

In [0]:
dbutils.library.restartPython()

In [0]:
!pip install python-dotenv

In [0]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
# Set environment variable
os.environ["OPENAI_API_KEY"] = openai_api_key

In [0]:
client = OpenAI(
    api_key= openai_api_key,
    base_url="https://dbc-a7bf52c7-eee8.cloud.databricks.com/serving-endpoints"
)

In [0]:
completion = client.chat.completions.create(
    model="databricks-meta-llama-3-1-8b-instruct",
    messages=[
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": "You are batman protector of Gotam city"
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "How is Gotham doing today?"
                }
            ]
        }
    ],
    max_tokens=5000
)

print(completion.choices[0].message.content)

Generating a custom python implementation

In [0]:
import mlflow
from mlflow import pyfunc

class BasicChatBot(pyfunc.PythonModel):
  def __init__(self,model_name:str):
      self.model_name = model_name
 
  def chatCompletionsAPI(self, user_query):
      openai_client = OpenAI(
        api_key= openai_api_key,
        base_url="https://dbc-a7bf52c7-eee8.cloud.databricks.com/serving-endpoints"
      )
      completion = openai_client.chat.completions.create(
          model = self.model_name,  
          messages=[
              {
                  "role": "system",
                  "content": [
                      {
                          "type": "text",
                          "text": "You are batman protector of Gotam city"
                      }
                  ]
              },
              {
                  "role": "user",
                  "content": [
                      {
                          "type": "text",
                          "text": user_query
                      }
                  ]
              }
          ],
          max_tokens=5000
      )
      return completion.choices[0].message.content  

  def predict(self,context,data):
          user_query = data["user_query"].iloc[0]
          gpt_response = self.chatCompletionsAPI(user_query)
          return gpt_response
  
     
          

In [0]:
test_model = BasicChatBot(model_name = "databricks-meta-llama-3-1-8b-instruct")

In [0]:
from mlflow.models import infer_signature
import pandas as pd
import mlflow

input_example = pd.DataFrame({"user_query":["How is Gotham doing today?"]})
output_example = pd.DataFrame({"gpt_response":["Gotham is facing cloudy eather with rising tensions downtown"]})



signature = infer_signature(input_example,output_example)
model_path = "basicchatbot"

mlflow.pyfunc.save_model(
    path = model_path,
    python_model = test_model,
    signature = signature,
    input_example = input_example
)

In [0]:
loaded_pyfunc_model = mlflow.pyfunc.load_model(model_path)

In [0]:
model_input = pd.DataFrame([{"user_query":"Hello How are you"}])
model_response = loaded_pyfunc_model.predict(model_input)
print(model_response)


In [0]:
import mlflow
run_id = None

with mlflow.start_run() as run:
    mlflow.log_artifacts(local_dir = model_path,artifact_path = "BasicChatBot")
    print(f" Model Logged with run ID: {run.info.run_id}")
    run_id = run.info.run_id


In [0]:
mlflow.register_model(f"runs:/{run_id}/BasicChatBot","BasicChatBot")